# LaughLM-Loop-135M TPU Pilot (PR #202 + PR #203)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dhiraj309/TrainLM/blob/main/notebooks/LaughLM_Loop_TPU_Pilot.ipynb)

End-to-end TPU pilot & ablation runner for `LaughLM-Loop-135M` using the hardened launch scripts (`stage1_base_30b.py`, `eval_loop_scaling.py`) and PR #202 + PR #203 features (`--max-steps None` token budget, `--shard-start`/`--shard-stop`, pre-import `/tmp/tpu_compile_cache`, `--deep-init-scale`, `--loop-depth-schedule`, `--doc-boundary-mask` via Pallas Splash `SegmentIds`, `--use-value-residual`, `--use-qk-norm`, `--use-attn-output-gate`).

### 1. Environment, Pre-Import Tier-3 Cache & TPU Discovery

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

# Export Tier-3 XLA/TorchTPU compile cache BEFORE importing torch or torch_tpu.
COMPILE_CACHE_DIR = Path("/tmp/tpu_compile_cache")
COMPILE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("TORCH_TPU_CACHE_DIR", str(COMPILE_CACHE_DIR))
os.environ.setdefault("TORCH_TPU_TIER3_COMPILATION_CACHE_ROOT", str(COMPILE_CACHE_DIR))
os.environ.setdefault("XLA_COMPILATION_CACHE_PATH", str(COMPILE_CACHE_DIR))

thp_path = Path("/sys/kernel/mm/transparent_hugepage/enabled")
if thp_path.exists():
    try:
        print("transparent_hugepage=", thp_path.read_text(encoding="utf-8").strip())
    except OSError as exc:
        print(f"transparent_hugepage=unreadable ({exc})")
else:
    print("transparent_hugepage=unavailable (container)")

BRANCH = "main"
REPO_URL = "https://github.com/Dhiraj309/TrainLM.git"
if (Path.cwd() / "scripts" / "laughlm" / "stage1_base_30b.py").is_file():
    REPO_DIR = Path.cwd()
else:
    REPO_DIR = Path("/content/TrainLM") if Path("/content").is_dir() else (Path.cwd() / "TrainLM")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "-e", "."], check=True)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "huggingface-hub>=1.5,<2",
        "PyYAML>=6,<7",
        "transformers>=5.15,<6",
        "datasets>=3,<5",
        "tokenizers>=0.21,<1",
    ],
    check=True,
)

import torch
if os.environ.get("PJRT_DEVICE") or sys.modules.get("torch_tpu") is not None:
    pass
try:
    import torch_tpu  # noqa: F401
except ModuleNotFoundError:
    print("torch_tpu not installed; running in non-TPU environment.")

from trainlm.runtime import discover_tpu

tpu_report = discover_tpu(torch, verify_allocation=False, include_properties=True)
print("repo_dir=", str(REPO_DIR))
print("tpu_compile_cache=", str(COMPILE_CACHE_DIR))
print("tpu_discovery=", json.dumps(tpu_report, indent=2, default=str))

### 2. Stage-1 Pilot Run (`scripts/laughlm/stage1_base_30b.py`)

In [ ]:
# Toggle PR #202 + PR #203 features (defaulting to full candidate ablation stack).
LOOP_PRESET = "pilot-1b"
DATASET_REPO = "Dhiraj45/laughlm-135m-shards-v2"
SUBFOLDER = "stage1"
SHARD_START = 0
SHARD_STOP = 1
MAX_TOKENS = 131_072_000
MICRO_BATCH_SIZE = 4
GRAD_ACCUM = 16
SAVE_STEPS = 250
LOGGING_STEPS = 25
COMPILE_CACHE_DIR = "/tmp/tpu_compile_cache"
OUTPUT_DIR = "runs/stage1_pilot-1b"

USE_DEEP_INIT_SCALE = True
LOOP_DEPTH_SCHEDULE = "2,4,4,6"
USE_DOC_BOUNDARY_MASK = True
USE_VALUE_RESIDUAL = True
USE_QK_NORM = True
USE_ATTN_OUTPUT_GATE = True

cmd = [
    sys.executable,
    "-u",
    "scripts/laughlm/stage1_base_30b.py",
    "--loop-preset", LOOP_PRESET,
    "--repo-id", DATASET_REPO,
    "--subfolder", SUBFOLDER,
    "--shard-start", str(SHARD_START),
    "--shard-stop", str(SHARD_STOP),
    "--compile-cache-dir", COMPILE_CACHE_DIR,
    "--max-tokens", str(MAX_TOKENS),
    "--micro-batch-size", str(MICRO_BATCH_SIZE),
    "--grad-accum", str(GRAD_ACCUM),
    "--save-steps", str(SAVE_STEPS),
    "--logging-steps", str(LOGGING_STEPS),
    "--output-dir", OUTPUT_DIR,
]
if USE_DEEP_INIT_SCALE:
    cmd.append("--deep-init-scale")
if LOOP_DEPTH_SCHEDULE:
    cmd.extend(["--loop-depth-schedule", LOOP_DEPTH_SCHEDULE])
if USE_DOC_BOUNDARY_MASK:
    cmd.append("--doc-boundary-mask")
if USE_VALUE_RESIDUAL:
    cmd.append("--use-value-residual")
if USE_QK_NORM:
    cmd.append("--use-qk-norm")
if USE_ATTN_OUTPUT_GATE:
    cmd.append("--use-attn-output-gate")

print("Running:", " ".join(cmd), flush=True)
proc = subprocess.run(cmd, check=False)
if proc.returncode != 0:
    raise RuntimeError(f"stage1_base_30b.py failed with exit code {proc.returncode}")

### 3. Held-Out Loop-Scaling Eval (`scripts/laughlm/eval_loop_scaling.py`, Shard `29..30`, `R=1..8`)

In [ ]:
EVAL_CHECKPOINT = "runs/stage1_pilot-1b/final_model"
EVAL_SHARD_START = 29
EVAL_SHARD_STOP = 30
EVAL_LOOPS = "1,2,4,6,8"
EVAL_JSON_PATH = Path("eval_loop_scaling.json")

eval_cmd = [
    sys.executable,
    "-u",
    "scripts/laughlm/eval_loop_scaling.py",
    "--checkpoint", EVAL_CHECKPOINT,
    "--repo-id", DATASET_REPO,
    "--subfolder", "stage1",
    "--eval-shard-start", str(EVAL_SHARD_START),
    "--eval-shard-stop", str(EVAL_SHARD_STOP),
    "--loops", EVAL_LOOPS,
    "--compile-cache-dir", COMPILE_CACHE_DIR,
    "--output-json", str(EVAL_JSON_PATH),
]
print("Running:", " ".join(eval_cmd), flush=True)
proc = subprocess.run(eval_cmd, check=False)
if proc.returncode != 0:
    raise RuntimeError(f"eval_loop_scaling.py failed with exit code {proc.returncode}")

eval_payload = json.loads(EVAL_JSON_PATH.read_text(encoding="utf-8"))
print("\n=== Held-Out Loop Scaling Summary ===")
print(f"{'R':>3s} | {'Loss':>8s} | {'PPL':>9s} | {'tok/s':>10s}")
print("-" * 40)
for row in eval_payload.get("results", []):
    print(f"{int(row['loops']):3d} | {row['loss']:8.4f} | {row['ppl']:9.2f} | {row['tokens_per_second']:10.1f}")

### 4. Qualitative Generation Check (`R=4`, TTFT & Decode `tok/s`)

In [ ]:
import importlib.util
import sys
import time
import torch
from transformers import AutoTokenizer
from trainlm.model import LaughLMLoopForCausalLM

if not torch.cuda.is_available() and importlib.util.find_spec("torch_tpu") is not None:
    import torch_tpu  # noqa: F401

CHECKPOINT_DIR = "runs/stage1_pilot-1b/final_model"
GEN_LOOPS = 4
MAX_NEW_TOKENS = 64
PROMPTS = (
    "Once upon a time",
    "The key difference between",
    "def fibonacci(n):",
    "In 1969, engineers",
)

def _sync(dev):
    if dev.type == "cpu":
        return
    accel = getattr(torch, "accelerator", None)
    if accel is not None and callable(getattr(accel, "synchronize", None)):
        accel.synchronize()

device = torch.device("cuda" if torch.cuda.is_available() else ("tpu" if "torch_tpu" in sys.modules else "cpu"))
dtype = torch.bfloat16 if device.type in {"cuda", "tpu"} else torch.float32
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT_DIR)
model = LaughLMLoopForCausalLM.from_pretrained(CHECKPOINT_DIR).to(device=device, dtype=dtype)
model.eval()

print(f"Generating on {device} ({dtype}) | R={GEN_LOOPS} | max_new_tokens={MAX_NEW_TOKENS}")
for prompt in PROMPTS:
    enc = tokenizer(prompt, return_tensors="pt")
    input_ids = enc["input_ids"].to(device)
    _sync(device)
    t0 = time.perf_counter()
    ttft_ms = None
    with torch.inference_mode():
        for step_idx in range(MAX_NEW_TOKENS):
            logits = model(input_ids=input_ids, num_loops=GEN_LOOPS).logits
            next_tok = logits[:, -1:, :].argmax(dim=-1)
            input_ids = torch.cat([input_ids, next_tok], dim=-1)
            if step_idx == 0:
                _sync(device)
                ttft_ms = (time.perf_counter() - t0) * 1000.0
    _sync(device)
    elapsed = max(1e-6, time.perf_counter() - t0)
    tok_s = MAX_NEW_TOKENS / elapsed
    text = tokenizer.decode(input_ids[0], skip_special_tokens=True)
    print(f"\n--- Prompt: {prompt!r} | TTFT={ttft_ms:.1f} ms | {tok_s:.1f} tok/s ---")
    print(text)